# Example 03 — sklearn Integration: Pipelines & GridSearchCV

`FuzzyARTMAP` is fully sklearn-compatible: it implements `get_params` /
`set_params`, `BaseEstimator`, and `ClassifierMixin`, so it plugs into
any sklearn workflow.

**Key concepts covered:**
- `Pipeline` with a custom normalisation step
- `cross_val_score` for robust evaluation
- `GridSearchCV` for hyperparameter search
- `clone` for safe reuse

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone

from fuzzyart import FuzzyARTMAP
from fuzzyart.preprocessing import normalize

## 1. Build a pipeline

In [ ]:
X, y = load_digits(return_X_y=True)
print(f'Digits dataset: {X.shape}  classes: {len(np.unique(y))}')

pipe = Pipeline([
    ('normalize', FunctionTransformer(normalize)),
    ('clf', FuzzyARTMAP(alpha=0.01, beta=0.5, epochs=5)),
])

## 2. Cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'CV F1 (weighted): {scores.mean():.3f} ± {scores.std():.3f}')

## 3. GridSearchCV — hyperparameter tuning

In [ ]:
param_grid = {
    'clf__alpha':        [0.001, 0.01, 0.1],
    'clf__beta':         [0.2, 0.5, 0.9],
    'clf__rho_baseline': [0.0, 0.1, 0.3],
}

gs = GridSearchCV(
    pipe, param_grid,
    cv=StratifiedKFold(3, shuffle=True, random_state=0),
    scoring='f1_weighted',
    n_jobs=-1, verbose=1
)
gs.fit(X, y)
print(f'Best score: {gs.best_score_:.3f}')
print(f'Best params: {gs.best_params_}')

## 4. Visualise hyperparameter landscape

In [ ]:
import pandas as pd

results = pd.DataFrame(gs.cv_results_)
pivot = results.pivot_table(
    index='param_clf__alpha',
    columns='param_clf__beta',
    values='mean_test_score'
)

plt.figure(figsize=(7, 4))
im = plt.imshow(pivot.values, aspect='auto', cmap='YlGnBu')
plt.colorbar(im, label='F1 (weighted)')
plt.xticks(range(len(pivot.columns)), [str(c) for c in pivot.columns])
plt.yticks(range(len(pivot.index)), [str(i) for i in pivot.index])
plt.xlabel('beta'); plt.ylabel('alpha')
plt.title('GridSearch — alpha vs beta')
plt.tight_layout(); plt.show()

## 5. Save and reload the best model

In [ ]:
best_clf = gs.best_estimator_.named_steps['clf']
best_clf.save('best_fuzzyartmap.pkl')

loaded = FuzzyARTMAP.load('best_fuzzyartmap.pkl')
X_norm = normalize(X)
preds = loaded.predict(X_norm)
print(f'Loaded model accuracy: {np.mean(preds == y):.3f}')